In [24]:
from langchain_ollama import ChatOllama

model = ChatOllama(model="gpt-oss:120b-cloud")
print('LLM is ready')

LLM is ready


In [ ]:
from langchain_core.prompts import PromptTemplate
from langchain_core.output_parsers import StrOutputParser
# Sequential chain example
prompt_1 = PromptTemplate(
    template="Which is the capital city of {country}?",
    input_variables=["country"]
)
prompt_2 = PromptTemplate(
    template="Name 5 famous tourist attractions in {capital_city}?",
    input_variables=["capital_city"]
)
str_parser = StrOutputParser()
sequential_chain = prompt_1 | model | str_parser | prompt_2 | model | str_parser
response = sequential_chain.invoke({"country": "France"})
print(response)

The capital city of France is actually Paris, and here are 5 famous tourist attractions in Paris:

1. The Eiffel Tower (La Tour Eiffel) - an iconic iron lattice tower built for the 1889 World's Fair and one of the most recognizable landmarks in the world.

2. The Louvre Museum (Musée du Louvre) - a world-renowned museum that houses an impressive collection of art and artifacts from ancient civilizations to the 19th century, including the Mona Lisa.

3. Notre-Dame Cathedral (Cathédrale Notre-Dame de Paris) - a beautiful Gothic cathedral built in the 12th century and one of the most famous churches in the world.

4. Arc de Triomphe - a monumental arch that honors the soldiers who fought and died for France, located in the center of the famous Champs-Élysées avenue.

5. The Champs-Élysées - a famous avenue lined with high-end boutiques, cafes, and theaters, and a popular destination for shopping and entertainment.

These are just a few of the many famous tourist attractions in Paris, but 

In [ ]:
from langchain_core.runnables import RunnableParallel
from langchain_core.prompts import PromptTemplate
from langchain_core.output_parsers import StrOutputParser
# Parallel chain example
prompt_1 = PromptTemplate(
    template="Which is the capital city of {country}?",
    input_variables=["country"]
)
prompt_2 = PromptTemplate(
    template="Name 5 famous tourist attractions in {country}?",
    input_variables=["country"]
)
str_parser = StrOutputParser()

parallel_chain = RunnableParallel  ({
    "capital_city": prompt_1 | model | str_parser,
    "tourist_attractions": prompt_2 | model | str_parser
})
response = parallel_chain.invoke({"country": "France"})
print(response)

{'capital_city': 'The capital city of France is Paris.', 'tourist_attractions': "Here are 5 famous tourist attractions in France:\n\n1. The Eiffel Tower (Paris) - An iconic iron lattice tower built for the 1889 World's Fair, it's one of the most recognizable landmarks in the world.\n\n2. The Louvre Museum (Paris) - A world-famous museum that houses an impressive collection of art and artifacts from ancient civilizations to the 19th century, including the Mona Lisa.\n\n3. The Palace of Versailles (Île-de-France) - A former royal palace with opulent decorations, gardens, and fountains, it's a testament to the excesses of the French monarchy.\n\n4. The Arc de Triomphe (Paris) - A monumental arch honoring the soldiers who fought and died for France, it offers stunning views of the city from its top.\n\n5. Notre-Dame Cathedral (Paris) - A beautiful Gothic cathedral built in the 12th century, it's one of the most famous landmarks in Paris and a symbol of French culture and history."}


In [10]:
# Example of sequential chains + parallel chains
prompt_3 = PromptTemplate(
    template="Write a summarizing paragraph about {capital_city} and more details about their tourist attractions {tourist_attractions}.",
    input_variables=["capital_city", "tourist_attractions"]
)
summary_chain = prompt_3 | model | str_parser
final_chain = parallel_chain | summary_chain
response = final_chain.invoke({"country": "France"})
print(response)

The capital city of France is indeed Paris, a city renowned for its rich history, art, and culture. Visitors to Paris can explore the iconic Eiffel Tower, the world-famous Louvre Museum, and the historic Notre-Dame Cathedral, among other famous landmarks. The Palace of Versailles, located just outside of Paris, is another must-visit destination, boasting opulent decorations, beautiful gardens, and an impressive Hall of Mirrors. For those looking to escape the city, the French Riviera offers stunning beaches, picturesque towns, and charming coastal villages like Nice, Cannes, and Saint-Tropez. With so many incredible tourist attractions, France is a top destination for travelers from around the world.


In [ ]:
# Example of conditional chain - Sentiment Analysis
from typing import Literal
from langchain_core.output_parsers import PydanticOutputParser
from pydantic import BaseModel, Field
from langchain_core.runnables import RunnableBranch, RunnableLambda

class FeedbackSentiment(BaseModel):
    sentiment: Literal["positive", "negative"] = Field(description="Sentiment of the feedback")

pydantic_parser = PydanticOutputParser(pydantic_object=FeedbackSentiment)

customer_feedback_prompt = PromptTemplate(
    template="Analyze the sentiment of the following customer feedback and classify it as positive or negative. feedback: \n{feedback}\n"
     " and provide the feedback in the following format {response_format}",
    input_variables=["feedback"],
    partial_variables={"response_format": pydantic_parser.get_format_instructions()}
)
sentiment_chain = customer_feedback_prompt | model | pydantic_parser

positive_prompt = PromptTemplate(
    template="Write a thank you note to the customer for their positive feedback: {feedback}",
    input_variables=["feedback"]
)
negative_prompt = PromptTemplate(
    template="Write an apology note to the customer for their negative feedback: {feedback}",
    input_variables=["feedback"]
)
positive_chain = positive_prompt | model | str_parser
negative_chain = negative_prompt | model | str_parser
default_chain = RunnableLambda(lambda x: "Thanks for your feedback. We will get back to you soon.")
# Conditional branching based on sentiment
branch = RunnableBranch(
    (lambda feedbackSentiment: feedbackSentiment['sentiment'].sentiment == "positive", positive_chain),
    (lambda feedbackSentiment: feedbackSentiment['sentiment'].sentiment == "negative", negative_chain),
    default_chain
)

#sentiment_result = sentiment_chain.invoke({"feedback": "I liked food of your hotel and staff was very helpful."})
#sentiment_result = sentiment_chain.invoke({"feedback": "Hotel rooms were clean and spacious."})
#email_to_customer = branch.invoke({"sentiment": sentiment_result, "feedback": "Hotelrooms were clean and spacious."})
sentiment_result = sentiment_chain.invoke({"feedback": "Hotel rooms were not clean"})
email_to_customer = branch.invoke({"sentiment": sentiment_result, "feedback": "Hotel rooms were not clean"})
#sentiment_result = sentiment_chain.invoke({"feedback": "No comments"})
#email_to_customer = branch.invoke({"sentiment": sentiment_result, "feedback": "No comments"})
print(email_to_customer)


**Subject: Our Sincere Apology for Your Recent Stay**

Dear [Guest Name],

I want to extend my deepest apologies for the condition of your hotel room during your recent stay with us. Cleanliness is a basic expectation and a core value at [Hotel Name], and it is clear that we fell short of both for you and for the standards we set for ourselves.

Your feedback is incredibly important, and we are taking immediate action to address the issues you raised:

1. **Full Review of Housekeeping Procedures** – Our housekeeping supervisor has been notified and will conduct a comprehensive audit of all cleaning protocols and checklists.  
2. **Additional Training** – All members of the housekeeping team will undergo refresher training focused on thoroughness, attention to detail, and room‑inspection standards.  
3. **Quality‑Control Spot Checks** – We are implementing random spot‑checks throughout the day to ensure every room meets our cleanliness standards before guests arrive.  
4. **Personal Fol